## TODO
- [X] GridSearchCV
- [X] Test PCA results
- [ ] Test with no PCA + SHAP [Feature Importance](https://www.geeksforgeeks.org/machine-learning/how-are-featureimportances-in-randomforestclassifier-determined/)
- [ ] PCA Feature Loadings
- [ ] Try SMOTE and SMOTE Tomek


# Thyroid Cancer Recurrence: Random Forest Classification
**Author:** Justin Williams  
**Date:** 11/3/2026

A demonstration of the Random Forest classification model. The notebook compares two evaluation methods: train/test splits and 5-fold cross-validation.

## Contents

1. Background
2. Setup and Constants
3. Load Data
4. Preprocessing
5. Model Pipeline
6. Evaluation
7. Results


## 1. Background

A `RandomForestClassifier` is an ensemble of decision trees. Each tree is trained on a random sample of rows and considers a random selection of features at each split. Each tree votes, and the majority vote is chosen for the final prediction.



## 2. Setup and Constants

In [18]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, 
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn import preprocessing
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold, 
    cross_val_predict, 
    train_test_split
)

In [19]:
TARGET_COL = "Recurred"
CATEGORICAL_COLS = [
    "Gender",
    "Smoking",
    "Hx Smoking",
    "Hx Radiotherapy",
    "Thyroid Function",
    "Physical Examination",
    "Adenopathy",
    "Pathology",
    "Focality",
    "Risk",
    "T",
    "N",
    "M",
    "Stage",
    "Response"
]

RAND_STATE = 42

TEST_SIZES = [0.25, 0.3, 0.35]

PARAM_DICT = {
    "clf__n_estimators": [80, 85, 90],
    "clf__max_features": ["sqrt", "log2"],
    "clf__max_depth": [10, 15, 20]
}

PCA_COMPONENTS = 21

pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", '{:.3f}'.format)

In [20]:
# Format a confusion matrix (cm) into a Pandas dataframe
def make_cm_frame(cm):
    return pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Pred No", "Pred Yes"]
    )

def make_load_frame(col_names, loadings):
    return pd.DataFrame(loadings)


### Cross-Validation Setup

`StratifiedKFold` keeps the ratio of target values consistent across each fold.

In [21]:
# Uses StratifiedKFold to keep the ratio of target values consistent across folds
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_STATE)

## 3. Load Data

In [22]:
df = pd.read_csv("../data/processed/Thyroid_Diff.csv")

print(df.shape)

df.head()

(383, 17)


,Age,Gender,Smoking,Hx Smoking,Hx Radiotherapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Age                   383 non-null    int64
 1   Gender                383 non-null    str  
 2   Smoking               383 non-null    str  
 3   Hx Smoking            383 non-null    str  
 4   Hx Radiotherapy       383 non-null    str  
 5   Thyroid Function      383 non-null    str  
 6   Physical Examination  383 non-null    str  
 7   Adenopathy            383 non-null    str  
 8   Pathology             383 non-null    str  
 9   Focality              383 non-null    str  
 10  Risk                  383 non-null    str  
 11  T                     383 non-null    str  
 12  N                     383 non-null    str  
 13  M                     383 non-null    str  
 14  Stage                 383 non-null    str  
 15  Response              383 non-null    str  
 16  Recurred           

## 4. Preprocessing

`X` holds the features to train on. `y` holds the target to predict. The target column maps Yes/No to 1/0.

In [24]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1})

print(y.value_counts())

Recurred
0    275
1    108
Name: count, dtype: int64


### Encoding and Scaling

Columns listed in `CATEGORICAL_COLS` are transformed using One-Hot Encoding. The `Age` column is scaled to have a mean of 0 and a standard deviation of 1. The `ColumnTransformer` is used to avoid data leakage from encoding training and testing data together. It produces the same columns, even if future data is missing any categories.


In [25]:
colTransform = ColumnTransformer(
    transformers=[
        ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", preprocessing.StandardScaler(), ["Age"])
    ],
    verbose_feature_names_out=False
)

## 5. Pipeline Setup

In [26]:
model = Pipeline([
    ("trans", colTransform),
    ("pca", PCA(n_components=PCA_COMPONENTS)),
    ("clf", RandomForestClassifier(random_state=RAND_STATE))
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trans', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{

### Hyperparameter Tuning - GridSearchCV

Adjust the parameters of the Random Forest classifier in the pipeline to find the combination of parameters with the best results. Fit to each train/test split.

In [27]:
gs = GridSearchCV(model, PARAM_DICT)

## 6. Evaluation

### Train-Test Splits

In [28]:
results = []
split_cms = {}

for size in TEST_SIZES:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=size, stratify=y, random_state=RAND_STATE
    )

    model.set_params(pca__n_components=21)
    gs = GridSearchCV(model, PARAM_DICT)

    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_pred)
    roc_auc = auc(fpr, tpr)

    pca = gs.best_estimator_.named_steps["pca"]
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

    results.append({
        "Method": f"Train/test split ({size})",
        "Report": classification_report(y_test, y_pred, target_names=["No", "Yes"], output_dict=True),
        "ROCAUC": roc_auc,
        "Best": gs.best_params_,
        "FeatLoad": loadings
    })
    split_cms[size] = confusion_matrix(y_test, y_pred)

### 5-Fold Cross-Validation

In [29]:
cv_pred = cross_val_predict(gs, X, y, cv=kfold)
cv_cm = confusion_matrix(y, cv_pred)

fpr, tpr, thresholds = roc_curve(y, cv_pred)
roc_auc = auc(fpr, tpr)
    
results.append({
    "Method": "5-fold cross-validation (mean)",
    "Report": classification_report(y, cv_pred, target_names=["No", "Yes"], output_dict=True),
    "ROCAUC": roc_auc,
    "Best": False,
    "FeatLoad": False
})

## 7. Results

### Classification Reports

In [30]:
for result in results:
    print(result["Method"])
    print(f"ROC AUC: {result["ROCAUC"]}")
    display(pd.DataFrame(result["Report"]))


Train/test split (0.25)
ROC AUC: 0.9412238325281804


,No,Yes,accuracy,macro avg,weighted avg
precision,0.971,0.893,0.948,0.932,0.949
recall,0.957,0.926,0.948,0.941,0.948
f1-score,0.964,0.909,0.948,0.936,0.948
support,69.000,27.000,0.948,96.000,96.000


Train/test split (0.3)
ROC AUC: 0.9506777108433735


,No,Yes,accuracy,macro avg,weighted avg
precision,0.976,0.909,0.957,0.942,0.957
recall,0.964,0.938,0.957,0.951,0.957
f1-score,0.970,0.923,0.957,0.946,0.957
support,83.000,32.000,0.957,115.000,115.000


Train/test split (0.35)
ROC AUC: 0.9370591427021161


,No,Yes,accuracy,macro avg,weighted avg
precision,0.960,0.944,0.956,0.952,0.955
recall,0.979,0.895,0.956,0.937,0.956
f1-score,0.969,0.919,0.956,0.944,0.955
support,97.000,38.000,0.956,135.000,135.000


5-fold cross-validation (mean)
ROC AUC: 0.9298989898989899


,No,Yes,accuracy,macro avg,weighted avg
precision,0.957,0.923,0.948,0.940,0.947
recall,0.971,0.889,0.948,0.930,0.948
f1-score,0.964,0.906,0.948,0.935,0.947
support,275.000,108.000,0.948,383.000,383.000


### Confusion Matrices

The rows represent the actual classes, and the columns represent predicted classes. All correct predictions lie on the diagonal:

| | Pred No | Pred Yes |
|---|---|---|
| **Actual No** | True negatives (TN) | False positives (FP) |
| **Actual Yes** | False negatives (FN) | True positives (TP) |

Train/Test Split matrices count the test set while the Cross-Validation matrix counts the entire dataset.

In [31]:
for size, cm in split_cms.items():
    print(f"Confusion matrix (test_size={size}):")
    display(make_cm_frame(cm))

print("Confusion matrix (5-fold CV):")
display(make_cm_frame(cv_cm))

Confusion matrix (test_size=0.25):


,Pred No,Pred Yes
Actual No,66,3
Actual Yes,2,25


Confusion matrix (test_size=0.3):


,Pred No,Pred Yes
Actual No,80,3
Actual Yes,2,30


Confusion matrix (test_size=0.35):


,Pred No,Pred Yes
Actual No,95,2
Actual Yes,4,34


Confusion matrix (5-fold CV):


,Pred No,Pred Yes
Actual No,267,8
Actual Yes,12,96


### Hyperparameter Tuning - GridSearchCV

The following hyper parameters for the Random Forest model were tuned:
1. `max_depth`
2. `max_features`
3. `n_estimators`


In [32]:
for result in results:
    print(result["Method"])
    print(result["Best"] if result["Best"] else "N/A")
    print()

Train/test split (0.25)
{'clf__max_depth': 10, 'clf__max_features': 'log2', 'clf__n_estimators': 90}

Train/test split (0.3)
{'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__n_estimators': 85}

Train/test split (0.35)
{'clf__max_depth': 20, 'clf__max_features': 'log2', 'clf__n_estimators': 90}

5-fold cross-validation (mean)
N/A



### Feature Loadings

Matrix of feature loadings, explaining the impact of each feature across each component.


In [33]:
for result in results:
    print(result["Method"])
    display(pd.DataFrame(result["FeatLoad"]) if result["Method"] != "5-fold cross-validation (mean)" else "N/A")

Train/test split (0.25)


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-0.185,0.005,0.033,0.142,0.176,-0.085,-0.040,0.133,-0.119,-0.009,...,-0.061,-0.046,0.006,0.019,-0.013,0.050,-0.063,0.011,0.007,-0.003
1,0.185,-0.005,-0.033,-0.142,-0.176,0.085,0.040,-0.133,0.119,0.009,...,0.061,0.046,-0.006,-0.019,0.013,-0.050,0.063,-0.011,-0.007,0.003
2,-0.175,-0.038,0.055,0.123,0.153,-0.063,-0.019,0.058,-0.014,-0.001,...,0.047,0.037,-0.013,-0.085,0.014,-0.039,0.110,-0.026,0.001,0.005
3,0.175,0.038,-0.055,-0.123,-0.153,0.063,0.019,-0.058,0.014,0.001,...,-0.047,-0.037,0.013,0.085,-0.014,0.039,-0.110,0.026,-0.001,-0.005
4,-0.042,-0.037,0.017,0.053,0.045,0.027,-0.026,0.031,0.026,0.100,...,0.032,0.077,0.055,0.049,0.121,-0.069,-0.058,0.027,0.024,-0.013
5,0.042,0.037,-0.017,-0.053,-0.045,-0.027,0.026,-0.031,-0.026,-0.100,...,-0.032,-0.077,-0.055,-0.049,-0.121,0.069,0.058,-0.027,-0.024,0.013
6,-0.029,-0.020,0.006,0.020,0.034,0.002,-0.011,-0.002,0.030,0.019,...,0.011,-0.003,0.010,-0.014,0.002,0.008,-0.031,0.029,0.016,0.036
7,0.029,0.020,-0.006,-0.020,-0.034,-0.002,0.011,0.002,-0.030,-0.019,...,-0.011,0.003,-0.010,0.014,-0.002,-0.008,0.031,-0.029,-0.016,-0.036
8,-0.001,0.024,0.031,-0.035,-0.018,-0.018,-0.003,-0.016,-0.045,-0.072,...,-0.016,-0.030,0.014,0.005,0.062,-0.034,0.019,0.003,-0.056,-0.036
9,-0.014,-0.006,0.006,0.000,-0.002,0.003,0.015,-0.020,-0.024,-0.038,...,-0.003,-0.030,-0.001,-0.015,0.035,-0.009,-0.024,0.014,0.018,-0.041


Train/test split (0.3)


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-0.171,0.011,0.006,0.003,-0.214,-0.102,-0.063,0.148,-0.111,-0.029,...,-0.044,0.048,0.000,0.037,0.002,0.039,0.054,0.026,0.009,-0.008
1,0.171,-0.011,-0.006,-0.003,0.214,0.102,0.063,-0.148,0.111,0.029,...,0.044,-0.048,-0.000,-0.037,-0.002,-0.039,-0.054,-0.026,-0.009,0.008
2,-0.167,-0.034,0.032,0.000,-0.191,-0.065,-0.044,0.055,-0.005,0.010,...,0.033,-0.047,0.000,-0.107,0.018,-0.013,-0.093,-0.045,-0.015,0.001
3,0.167,0.034,-0.032,-0.000,0.191,0.065,0.044,-0.055,0.005,-0.010,...,-0.033,0.047,-0.000,0.107,-0.018,0.013,0.093,0.045,0.015,-0.001
4,-0.035,-0.038,0.009,0.017,-0.074,0.006,0.029,0.047,0.036,0.095,...,0.050,-0.044,-0.052,0.026,-0.156,-0.049,0.034,0.006,0.020,-0.011
5,0.035,0.038,-0.009,-0.017,0.074,-0.006,-0.029,-0.047,-0.036,-0.095,...,-0.050,0.044,0.052,-0.026,0.156,0.049,-0.034,-0.006,-0.020,0.011
6,-0.031,-0.021,0.004,-0.004,-0.046,-0.008,0.004,-0.000,0.031,0.023,...,0.008,0.007,-0.012,-0.010,-0.006,0.009,0.025,0.041,0.008,0.041
7,0.031,0.021,-0.004,0.004,0.046,0.008,-0.004,0.000,-0.031,-0.023,...,-0.008,-0.007,0.012,0.010,0.006,-0.009,-0.025,-0.041,-0.008,-0.041
8,-0.000,0.027,0.024,-0.020,0.037,-0.020,-0.001,-0.017,-0.050,-0.024,...,-0.010,0.041,0.012,-0.012,-0.047,-0.021,-0.022,-0.006,-0.053,-0.077
9,-0.015,-0.006,0.006,0.001,0.001,0.010,-0.007,-0.023,-0.034,-0.003,...,-0.013,0.037,0.020,-0.023,-0.036,0.004,0.015,0.017,0.018,-0.004


Train/test split (0.35)


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,-0.147,0.006,0.008,0.044,-0.232,-0.060,-0.075,0.155,-0.058,-0.047,...,-0.030,0.057,0.016,0.037,0.013,0.032,-0.054,-0.050,0.003,-0.013
1,0.147,-0.006,-0.008,-0.044,0.232,0.060,0.075,-0.155,0.058,0.047,...,0.030,-0.057,-0.016,-0.037,-0.013,-0.032,0.054,0.050,-0.003,0.013
2,-0.142,-0.032,0.033,0.035,-0.190,-0.046,-0.035,0.072,0.009,0.029,...,0.012,-0.057,-0.006,-0.109,-0.034,0.002,0.063,0.072,-0.002,0.019
3,0.142,0.032,-0.033,-0.035,0.190,0.046,0.035,-0.072,-0.009,-0.029,...,-0.012,0.057,0.006,0.109,0.034,-0.002,-0.063,-0.072,0.002,-0.019
4,-0.017,-0.031,0.015,0.032,-0.046,0.003,0.038,0.044,-0.013,0.093,...,0.031,-0.047,-0.073,0.014,0.147,-0.083,0.005,-0.027,0.023,-0.000
5,0.017,0.031,-0.015,-0.032,0.046,-0.003,-0.038,-0.044,0.013,-0.093,...,-0.031,0.047,0.073,-0.014,-0.147,0.083,-0.005,0.027,-0.023,0.000
6,-0.003,-0.007,0.004,0.003,-0.013,-0.008,-0.003,-0.005,0.005,0.012,...,0.006,0.008,-0.008,-0.010,0.008,-0.004,-0.007,-0.019,-0.006,0.005
7,0.003,0.007,-0.004,-0.003,0.013,0.008,0.003,0.005,-0.005,-0.012,...,-0.006,-0.008,0.008,0.010,-0.008,0.004,0.007,0.019,0.006,-0.005
8,-0.016,0.007,0.023,-0.021,0.003,-0.014,0.007,-0.004,-0.023,-0.011,...,0.005,0.027,0.017,-0.012,0.034,-0.017,0.018,-0.002,-0.031,-0.044
9,-0.017,-0.006,0.007,0.001,-0.004,0.013,-0.001,-0.022,-0.031,-0.008,...,-0.010,0.043,0.044,-0.029,0.047,-0.010,0.008,-0.015,0.020,0.095


5-fold cross-validation (mean)


'N/A'